This is an ephemeral script that was relevant only at the time it was used.

The point of this script is to understand what came of the generations' judgements that we ran last weekend (12th through 14th). Every single job crashed. One due to Cuda OOM, one due to unknown reasons.

In [1]:
import json
import orjson
import tqdm
from pathlib import Path

folder = Path.cwd() / "outputs_gemma9b_judging_generations_partial_crash3"
subfolders = [f for f in folder.iterdir() if f.is_dir()]
error_infos = []
for subfolder in subfolders:
    error_file = subfolder / "error.json"
    error_contents = orjson.loads(error_file.read_bytes())
    is_error = not error_contents["success"]
    # print(is_error)
    config_file = subfolder / "config.json"
    config_contents = None
    if config_file.exists():
        config_contents = orjson.loads(config_file.read_bytes())
    if not error_contents["success"]:
        error_infos.append(
            {
                "config": config_contents,
                "is_error": is_error,
                "error_contents": error_contents,
                "subfolder_name": Path(subfolder).name,
            }
        )
print("Total number of errors:", len(error_infos), "out of", len(subfolders))
print("Should be 0 out of 29")
error_subfolder_names = set([error_info["subfolder"] for error_info in error_infos])

Total number of errors: 0 out of 29
Should be 0 out of 29


In [2]:
for error_info in error_infos:
    print(json.dumps(error_info, indent=4))

In [3]:
# Sanity check everyone else has the same queries
from transformers import AutoTokenizer

queries_set = None
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-9b-it")
for subfolder in tqdm.tqdm(subfolders, desc="Checking everyone else"):
    if subfolder.name in error_subfolder_names:
        continue
    generations_file = subfolder / "generations.jsonl"
    contents = [
        orjson.loads(line) for line in generations_file.read_bytes().splitlines()
    ]
    messages = [c["messages"] for c in contents]
    assert all(m[-1]["role"] == "assistant" for m in messages)
    types = [c["type"] for c in contents]
    messages_requests_chat_templatted = [
        tokenizer.apply_chat_template(
            # Exclude the last message
            m[:-1],
            tokenize=False,
            add_generation_prompt=True,
        )
        for m in messages
    ]
    hashables: list[tuple[str, str]] = [
        (_type, chat_templatted_message)
        for _type, chat_templatted_message in zip(
            types, messages_requests_chat_templatted
        )
    ]
    if queries_set is None:
        queries_set = set[tuple[str, str]](hashables)
    else:
        assert queries_set == set(hashables)

/mnt/align4_drive2/adrianoh/miniconda3/envs/scopebench/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking everyone else: 100%|██████████| 29/29 [00:00<00:00, 119.53it/s]


Conclusion is that we are missing just one set of generations, we should try to generate this one and then we will have all the requisite generations! This was later implemented as `--retry` flag in `script_2025_12_12_judging_checkpoints_do_generation.py`